# **Pre-Trained Model 1 - Efficient Net B0**

EfficientNetB0 was selected as the first pretrained architecture for the modelling phase. <br>


### **Why EfficientNet?**

EfficientNet is a CNN (convolutional neural network) built upon the concept of *compound scaling* — jointly scaling network depth, width and resolution via a fixed coefficient, rather than scaling any one dimension in isolation. This yields significantly better accuracy-per-parameter than other architectures.

### **Why B0 specifically?**

B0 is the baseline of the EfficientNet family (φ = 0). With only approximatelly 5M parameters it is the lightest variant, balacing **speed and accuracy**, making it ideal for a constrained medical dataset (~11,000 images). We believe that a heavier variant (B4–B7) would risk overfitting without substantially more data or aggressive regularisation. Therefore, B0 sits at the **optimal point on the bias–variance trade-off for this task**.

#### Moreover, we took into consideration even more specific dataset characteristics for these choice:
- **Suitability for dermoscopy**: Dermoscopic classification relies on fine-grained texture and colour pattern recognition (such as, atypical pigment networks, regression structures and vascular patterns). EfficientNet's depthwise separable convolutions and MBConv blocks are effective at capturing both local texture detail and global spatial context;
- **Transfer learning rationale**: ImageNet pre-training provides a rich, low-level feature initialisation (edges, textures, blobs) that are important for these medical images. Fine-tuning only the top layers in phase 1 prevents catastrophic forgetting of these representations and lets the new classification head converge stably.


### Imports

In [ ]:
import tensorflow as tf
#prevent TF from grabbing all GPU memory at once
gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

In [ ]:
from tensorflow.keras import mixed_precision

# Tell Keras to use float16 for memory, but keep float32 for numeric stability in the loss
policy = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(policy)

print('Compute dtype: %s' % policy.compute_dtype)
print('Variable dtype: %s' % policy.variable_dtype)

In [ ]:
import os
import sys
import keras.applications
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.preprocessing import label_binarize
from keras.applications.efficientnet import preprocess_input


if os.getcwd().endswith('models'):
    os.chdir('..')
    
from utils.utils_model import *
from utils.utils_augmentation import *
from utils.utils_preproc import *

In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
print("GPUs available:", gpus)
print("TF built with CUDA:", tf.test.is_built_with_cuda())
print("GPU available to TF:", tf.test.is_gpu_available())  # deprecated but still works

### **Data** Configuration

In [ ]:
# Load label mapping first — everything else depends on it
with open("label2idx.json", "r") as f:
    label2idx = json.load(f)

N_CLASSES  = len(label2idx)
BATCH_SIZE = 32

train_df = pd.read_csv('data/augmented_metadata.csv')
val_df   = pd.read_csv('data/val_split.csv')
test_df  = pd.read_csv('data/test_split.csv')

# Normalise column names and encode labels for all splits
for df in [train_df, val_df, test_df]:
    if 'cleaned_path' in df.columns and 'image_path' in df.columns:
        df.drop(columns=['image_path'], inplace=True)
    df.rename(columns={'cleaned_path': 'image_path'}, inplace=True)
    df['dx_encoded'] = df['dx'].map(label2idx).astype(int)

# Build datasets
train_ds = make_dataset(train_df, shuffle=True, repeat=True, preprocess_fn=preprocess_input)
val_ds   = make_dataset(val_df, preprocess_fn=preprocess_input)
test_ds  = make_dataset(test_df, preprocess_fn=preprocess_input)

STEPS_PER_EPOCH = len(train_df) // BATCH_SIZE
class_weights_dict = make_class_weights(train_df)

In [ ]:
print(f"Steps per epoch: {STEPS_PER_EPOCH}")
print(f"Class weights: {class_weights_dict}")

In [ ]:
def categorical_focal_loss(alpha=0.25, gamma=2.0):
    def focal_loss(y_true, y_pred):
        # Convert to float and clip to prevent log(0)
        y_true = tf.cast(y_true, tf.int32)
        y_true = tf.one_hot(y_true, depth=7) 
        y_pred = tf.clip_by_value(y_pred, tf.keras.backend.epsilon(), 1.0 - tf.keras.backend.epsilon())
        
        # Calculate cross entropy
        cross_entropy = -y_true * tf.math.log(y_pred)
        
        # Calculate weight factor
        loss = alpha * tf.math.pow(1 - y_pred, gamma) * cross_entropy
        return tf.reduce_sum(loss, axis=-1)
    return focal_loss

### **Model** Configuration

In [ ]:
def categorical_focal_loss(alpha=0.25, gamma=2.0):
    """
    Multiclass Focal Loss for sparse integer labels.
    alpha: Weighting factor to balance classes.
    gamma: Focusing parameter to prioritize hard samples.
    """
    def focal_loss(y_true, y_pred):
        # Convert sparse labels to one-hot for math
        y_true = tf.one_hot(tf.cast(y_true, tf.int32), depth=7)
        
        # Clip to prevent log(0) and stabilize training
        y_pred = tf.clip_by_value(y_pred, tf.keras.backend.epsilon(), 1.0 - tf.keras.backend.epsilon())
        
        # Calculate cross entropy
        cross_entropy = -y_true * tf.math.log(y_pred)
        
        # Apply the 'Focusing' factor
        loss = alpha * tf.math.pow(1 - y_pred, gamma) * cross_entropy
        
        return tf.reduce_sum(loss, axis=-1)
    return focal_loss

In [ ]:
#teste1
def build_efficientnet():
    base = keras.applications.EfficientNetB0(
        include_top=False,
        weights="imagenet",
        input_shape=(224, 224, 3),
        pooling=None
        
    )
    base.trainable = False

    inputs = keras.Input(shape=(224, 224, 3))
    x = base(inputs, training=False)

    x = keras.layers.GlobalAveragePooling2D()(x)
    x = keras.layers.Dropout(0.5)(x)                                      # was 0.4
    x = keras.layers.Dense(256, activation="relu", kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Dropout(0.4)(x)                                      # was 0.3
    outputs = keras.layers.Dense(N_CLASSES)(x)
    return keras.Model(inputs, outputs, name="efficientnet_b0")

In [ ]:
#teste2
def build_efficientnet():
    # Base model
    base = keras.applications.EfficientNetB0(
        include_top=False, weights='imagenet', input_shape=(224, 224, 3)
    )
    base.trainable = False 

    inputs = keras.Input(shape=(224, 224, 3))
    x = base(inputs, training=False) 

    # Dual Pooling
    gap = keras.layers.GlobalAveragePooling2D()(x)
    gmp = keras.layers.GlobalMaxPooling2D()(x)
    combined = keras.layers.Concatenate()([gap, gmp])
    
    # Head with 128 units and 0.6 Dropout
    x = keras.layers.BatchNormalization()(combined)
    x = keras.layers.Dense(128, activation="relu", kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    x = keras.layers.Dropout(0.6)(x)
    
    # Output layer using Softmax for Focal Loss compatibility
    outputs = keras.layers.Dense(N_CLASSES, activation="softmax")(x)
    
    return keras.Model(inputs, outputs)

In [ ]:
# phase 1: head only
model_pt1 = build_efficientnet() # Model Pre-Trained 1
model_pt1.summary()

model_pt1.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=categorical_focal_loss(alpha=0.25, gamma=2.0),
    metrics=["accuracy", BalancedAccuracy(N_CLASSES)]
)

my_callbacks = get_callbacks(
    checkpoint_path="checkpoints/model_EN_phase1.weights.h5",
    model=model_pt1,      # <--- Pass your model here
    max_diff=0.25,        # <--- Your 15% limit
    patience_es=10, 
    patience_lr=5
)

history1_pt1 = model_pt1.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    steps_per_epoch=STEPS_PER_EPOCH,
    class_weight=class_weights_dict,
    callbacks=my_callbacks
)

plot_history(history1_pt1, "Model B — EfficientNetB0 Phase 1 (head only)")

In [ ]:
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy('mixed_float16')
print("Policy:", mixed_precision.global_policy().name)

In [ ]:
# Load the Phase 1 weights from the hard drive
model_pt2 = build_efficientnet()
model_pt2.load_weights("checkpoints/model_EN_phase1.weights.h5")
print("Phase 1 weights loaded successfully!")

In [ ]:
# 1. Unfreeze ONLY the top blocks (from block6 onward)
base_b = model_pt2.get_layer('efficientnetb0')
base_b.trainable = True

# We freeze everything, then surgically unfreeze the top
# EfficientNetB0 has blocks 1 through 7. Unfreezing 6 and 7 is usually optimal.
unfreeze_from = "block6a_expand_conv" 
found = False

for layer in base_b.layers:
    if layer.name == unfreeze_from:
        found = True
    
    if not found:
        layer.trainable = False
    else:
        # Crucial: Keep BatchNormalization layers frozen even in unfrozen blocks
        if isinstance(layer, keras.layers.BatchNormalization):
            layer.trainable = False

model_pt2.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss=categorical_focal_loss(alpha=0.25, gamma=2.0),
    metrics=["accuracy", BalancedAccuracy(N_CLASSES)])

my_callbacks = get_callbacks(
    checkpoint_path="checkpoints/model_EN_phase2.weights.h5",
    model=model_pt2,      # <--- Pass your model here
    max_diff=9999,        # <--- Your 15% limit
    patience_es=6, 
    patience_lr=3
)

history2_b0 = model_pt2.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    steps_per_epoch=STEPS_PER_EPOCH,
    class_weight=class_weights_dict,
    callbacks=my_callbacks
)

plot_history(history2_b0, "Model B — EfficientNetB0 Phase 2 (fine-tune)")
results_b = evaluate_model(model_pt2, test_ds, label2idx, "Model B — EfficientNetB0")

In [ ]:
#394/394 [==============================] - 67s 169ms/step - loss: 0.3377 - accuracy: 0.8473 - balanced_accuracy: 0.8777 - val_loss: 0.6139 - val_accuracy: 0.7752 - val_balanced_accuracy: 0.6575 - lr: 2.5000e-06

### Model Evaluation